In [ ]:
import geopandas as gpd
import osmnx as ox
import polars as pl
from pyproj import Transformer
from valhalla import Actor, get_config

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

ox.settings.log_console = False
ox.settings.use_cache = True

In [ ]:
# def read_street_edge_list(device_group: int, path: str) -> list:
#     print(f"{device_group:02x}")
#     try:
#         with open(path, "br") as fp:
#             street_edge_list = pickle.load(fp)
#     except FileNotFoundError:
#         print(f"{path} was not found, generating it...")
#         df = prepare_parquet(device_group, days=opts.days, dir=opts.dir)
#         # runs more than 15 min on a full partition
#         street_edge_list = ox.distance.nearest_edges(
#             g_, X=df.geometry.x, Y=df.geometry.y
#         )  # , interpolate=25)
#         with open(path, "bw") as fp:
#             pickle.dump(street_edge_list, fp)
#     return street_edge_list

In [10]:
config = get_config(
    tile_extract="../data/valhalla/hungary/valhalla_tiles.tar",
    tile_dir="../data/valhalla/hungary/",
)
actor = Actor(config)

In [17]:
DIR = "/mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest/partitioned/"

In [18]:
for i in range(1):
    df = pl.read_parquet(f"{DIR}/{i:02x}/pings.parquet")

In [ ]:
# actor.trace_route()

In [ ]:
# request = {
#     "shape": [
#         {"lat": 47.47992, "lon": 19.074155, "type": "break"},
#         {"lat": 47.479995, "lon": 19.074221, "type": "via"},
#         {"lat": 47.476752, "lon": 19.075154, "type": "break"},
#     ],
#     "costing": "auto",
#     # "shape_match": "edge_walk",
# }

# result = actor.trace_route(request)

In [ ]:
# request = {
#     "shape": [
#         {"lat": 47.47992, "lon": 19.074155, "type": "break"},
#         {"lat": 47.479995, "lon": 19.074221, "type": "via"},
#         {"lat": 47.476752, "lon": 19.075154, "type": "break"},
#     ],
#     "costing": "auto",
#     # "shape_match": "edge_walk",
# }

# result = actor.trace_attributes(request)

In [3]:
area = gpd.read_file("../data/osm/budapest/boundary.geojson")

In [39]:
g_drive = ox.graph_from_polygon(area.geometry[0], network_type="drive", simplify=False)
# 2m 1s

ox.save_graphml(g_drive, "../output/budapest/drive.graphml")
# 7s

In [40]:
g_all = ox.graph_from_polygon(area.geometry[0], network_type="all", simplify=False)
# 3m 42s

ox.save_graphml(g_all, "../output/budapest/all.graphml")
# 1m

In [41]:
g_all_proj = ox.project_graph(g_all, to_crs=3857)

In [24]:
xs, ys = transformer.transform(
    df["longitude"].to_numpy(),
    df["latitude"].to_numpy(),
)

df_proj = df.with_columns(
    pl.Series("lon", xs),
    pl.Series("lat", ys),
)

In [42]:
street_edge_list = ox.distance.nearest_edges(
    g_all_proj,
    X=df_proj["lon"],
    Y=df_proj["lat"],
    return_dist=True,
)
# 7m 45s

In [31]:
street_edge_list

(array([(14032653687, 369824812, 0), (14032653687, 369824812, 0),
        (12534053804, 13276990508, 0), ..., (12482302541, 12482302540, 0),
        (12482302541, 12482302540, 0), (12482302541, 12482302540, 0)],
       shape=(15460994,), dtype=object),
 array([ 71.81249249,  74.18688149,   8.373973  , ..., 645.82565481,
        643.66515372, 648.51461667], shape=(15460994,)))

In [32]:
street_edge_list[0]

array([(14032653687, 369824812, 0), (14032653687, 369824812, 0),
       (12534053804, 13276990508, 0), ..., (12482302541, 12482302540, 0),
       (12482302541, 12482302540, 0), (12482302541, 12482302540, 0)],
      shape=(15460994,), dtype=object)

In [ ]:
street_edge_list[1]

In [34]:
len(street_edge_list[0])

15460994

In [35]:
len(df_proj)

15460994

In [ ]:
df_proj

In [43]:
osmids = [g_all.edges[u, v, key]["osmid"] for u, v, key in street_edge_list[0]]

In [44]:
osmids

[25702955,
 25702955,
 577189673,
 25702955,
 577189673,
 25702955,
 577189673,
 25702955,
 25702955,
 261373301,
 54724399,
 35775318,
 1220449594,
 35775318,
 663022305,
 459647990,
 474449785,
 346277125,
 1329683630,
 548422398,
 548422397,
 458338077,
 848995617,
 848995617,
 272986118,
 848995617,
 272986118,
 848995617,
 272986118,
 272986118,
 272986118,
 272986118,
 272986118,
 154141667,
 405163097,
 405163097,
 405163097,
 272986118,
 272986118,
 272986118,
 1133079652,
 26377845,
 272986118,
 272986118,
 26377845,
 848995617,
 848995617,
 1036300873,
 26136319,
 26136319,
 1133079652,
 26136319,
 756452686,
 1532681963,
 474449785,
 311254910,
 184015418,
 1147450677,
 489635601,
 10221485,
 73031467,
 663022308,
 7988817,
 165713723,
 608833576,
 395458141,
 26377845,
 848995617,
 848995617,
 848995617,
 848995617,
 848995617,
 848995617,
 554989548,
 25702955,
 25702955,
 25702955,
 1133079652,
 25702961,
 1133079652,
 1133079652,
 1133079652,
 577189673,
 25702955,
 2613

In [47]:
df.with_columns(street=osmids)

: 